# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shwetabh1013/flyrank-ml-internship-shwetabh/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Ranking / scoring.**

Lane 2 (Refresh / Content Opportunity Scoring) answers "which ones first?", not "will this one decline?" or "what groups exist?". The editor doesn't need a yes/no per page — they need an ordered queue: given a fixed sprint budget (N editor-hours), which pages should sit at the top. That maps directly to the ranking/scoring row in the framing table, not classification (no hard cutoff makes sense when the real constraint is a limited number of review slots, not a threshold) and not clustering (I'm not discovering content archetypes, I already have a specific action — refresh — in mind).


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target (provisional):** a continuous `priority_score` per content item that combines three observed ingredients — staleness (`days_since_last_update`), demand (`search_volume`), and an observed recent trend (the measured delta between `clicks_last_30d` and `clicks_prev_30d`). Rank pages by this score; the top N per client become the sprint's refresh queue.

**Where it comes from, honestly:**
- `days_since_last_update` and `search_volume` are directly observed fields — no proxy problem there.
- The click-delta trend is also observed (two real 30-day windows, not a label).
- What I am **not** doing: using `trend_direction` or `trend_pct` as a feature or as the target itself. Per the flyrank-data skill's label trap, `trend_direction` is *derived from* `trend_pct`, so treating it as ground truth would mean my "prediction" is just re-deriving a rule that's already in the data — not learning anything.
- This week's score is a **hand-weighted proxy**, not a trained target. The real target — did the page's traffic actually recover in the *N* weeks *after* a refresh — doesn't exist yet in the starter CSV; it needs the warehouse's daily table and a proper future time window. That's the Week 4-5 swap flagged in Week 1: replace `trend_direction` with an observed future-window outcome once I can build one.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**precision@K**, computed per client, where K = a realistic sprint size (I'll use K=20 as a stand-in editor budget).

Concretely: of the top 20 pages my score surfaces for a client, what fraction are pages that are *both* stale *and* observed declining (`trend_direction == 'down'`, used here only as a descriptive check label, never as a model input)? That's the number I can compute today against a baseline, and it's the number an editor would actually feel — "of the 20 pages I worked this sprint, how many were real."

I'm choosing precision@K over a global metric like RMSE or overall correlation because the cost structure from Week 1 is precision-conscious at the top: an editor only ever sees the head of the queue, so what happens at rank 5,000 doesn't matter nearly as much as what happens in the top 20. "Good" here means beating the simplest possible baseline (rank by staleness alone) — which, as the code below shows, my first hand-weighted attempt did not manage. That's the metric doing its job: it caught a bad guess before I trusted it.


In [1]:
import pandas as pd
from pathlib import Path

candidates = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
]
csv_path = next(p for p in candidates if p.exists())
df = pd.read_csv(csv_path)

# hand-weighted proxy score: staleness + demand + observed trend, all min-max scaled
def minmax(s):
    return (s - s.min()) / (s.max() - s.min())

click_delta = df["clicks_prev_30d"] - df["clicks_last_30d"]  # positive = losing clicks

score_df = df.copy()
score_df["priority_score"] = (
    0.4 * minmax(score_df["days_since_last_update"].fillna(0))
    + 0.4 * minmax(score_df["search_volume"].fillna(0))
    + 0.2 * minmax(click_delta.clip(lower=0))
)

# baseline for comparison: a plain rule, "stale only"
score_df["is_stale_rule"] = score_df["freshness_tier"].isin(["91-180", "181+"])

def precision_at_k(sub, k=20):
    top_k = sub.sort_values("priority_score", ascending=False).head(k)
    hits = ((top_k["freshness_tier"].isin(["91-180", "181+"])) & (top_k["trend_direction"] == "down")).sum()
    return hits / k

def precision_at_k_rule(sub, k=20):
    # baseline: rank by days_since_last_update alone (the "just flag anything stale" rule)
    top_k = sub.sort_values("days_since_last_update", ascending=False).head(k)
    hits = ((top_k["freshness_tier"].isin(["91-180", "181+"])) & (top_k["trend_direction"] == "down")).sum()
    return hits / k

clients = score_df["client_id"].unique()
score_precisions = [precision_at_k(score_df[score_df["client_id"] == c]) for c in clients]
rule_precisions = [precision_at_k_rule(score_df[score_df["client_id"] == c]) for c in clients]

print(f"{len(clients)} clients")
print(f"precision@20, weighted score:  {sum(score_precisions)/len(score_precisions):.2f}")
print(f"precision@20, stale-only rule: {sum(rule_precisions)/len(rule_precisions):.2f}")


32 clients
precision@20, weighted score:  0.29
precision@20, stale-only rule: 0.31


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one pseudonymized content item (a single URL/page) belonging to one client, described by its trailing-90-day performance. `content_id` + `client_id` together uniquely identify a row (30,000 rows, 30,000 unique `(content_id, client_id)` pairs, confirmed below) — both are pseudonyms used only to group and split, never as features, per the flyrank-data skill.


In [2]:
# confirm the grain: one row per (content_id, client_id)
n_rows = len(df)
n_unique_pairs = df[["content_id", "client_id"]].drop_duplicates().shape[0]
print(f"rows: {n_rows:,}  |  unique (content_id, client_id) pairs: {n_unique_pairs:,}  |  grain holds: {n_rows == n_unique_pairs}")

# show the actual slice a scoring model would consume for this lane
lane_cols = [
    "content_id", "client_id", "content_type", "freshness_tier",
    "days_since_last_update", "search_volume", "clicks_last_30d",
    "clicks_prev_30d", "trend_direction",
]
df[lane_cols].head(5)


rows: 30,000  |  unique (content_id, client_id) pairs: 30,000  |  grain holds: True


,content_id,client_id,content_type,freshness_tier,days_since_last_update,search_volume,clicks_last_30d,clicks_prev_30d,trend_direction
0,content_304f48230142,client_f369cb89fc,keyword article,0-30,20,10.0,2,13,down
1,content_a1fb4e703a9e,client_4e07408562,keyword article,0-30,25,90.0,2,1,down
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,0-30,20,0.0,1,3,down
3,content_331d6c4de07b,client_19581e27de,keyword article,0-30,22,10.0,22,17,stable
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,0-30,14,0.0,10,2,down


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**Honest result first:** my hand-weighted score (0.4 staleness + 0.4 demand + 0.2 trend) scored precision@20 = 0.29, actually *below* the plain stale-only rule's 0.31. I'm reporting that, not hiding it — it's the actual finding, and it's the argument, not a failure of the argument.

That result is exactly why a single if-statement (or a single hand-guessed formula, which is just a fancier if-statement) isn't enough here. Guessing 0.4/0.4/0.2 was no different in kind from guessing a stale-only threshold — both are a human picking numbers without checking them against an outcome. The signals interact in ways a flat weighted sum can't capture: the content-type breakdown above shows `trend_direction == 'down'` ranges from 28.7% (feedly article) to 57.2% (comparison article) — a fixed global weight on "trend" means the same number is either too aggressive or too conservative depending on content type, and my hand-picked 0.2 was wrong in a way I could not have known without testing it.

That's the real case for ML over a fixed rule: not that ML is inherently smarter, but that the right weights and interactions have to be *learned against an observed outcome*, not asserted. This week I don't have that outcome yet — my precision@20 check above uses `trend_direction`, itself a proxy, only as a descriptive sense-check, never as a training signal. The task for Weeks 4-5 is to build a real future-window label from the warehouse's daily table and let a model find weights that a person guessing by hand — including me, today — got wrong.


In [3]:
# quick evidence the tangle is real: does trend direction depend on client/content_type,
# i.e. is a single global threshold going to be wrong for someone?
import numpy as np

pivot = (
    df.groupby("content_type")["trend_direction"]
      .value_counts(normalize=True)
      .unstack()
      .round(3)
)
pivot


trend_direction,down,flat,new,stable,up
content_type,,,,,
comparison article,0.572,NaN,0.003,0.235,0.189
feedly article,0.287,0.052,0.466,0.093,0.102
keyword article,0.561,0.038,0.046,0.206,0.149


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.